# 🎵 Music Generation — Preprocessing & 12+ Block EDA Pipeline
**CSE425 Neural Networks | Ummay Maimona Chaman | 22301719 | Section 1**

This notebook provides a comprehensive exploratory data analysis of the REAL MIDI datasets (MAESTRO, Lakh, Groove) and visualizes the performance of all 4 Neural Network models.

### Pipeline Overview:
1. **EDA**: 12+ statistical analysis blocks of the training data.
2. **Latent Analysis**: Visualizing model embeddings using PCA and Clustering.
3. **Training Progress**: All 4 task loss curves.
4. **Final Comparison**: Global performance metrics including Loss, Perplexity, and Genre Control.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pandas as pd
from IPython.display import Image, display

from src.config import *
from src.preprocessing.midi_parser import NoteEvent

plt.style.use('seaborn-v0_8-bright')
sns.set_palette('bright')
print('Setup complete!')

## 1. Exploratory Data Analysis (EDA)
We analyze the statistical properties of the REAL data processed in `data/train_test_split`.

In [ ]:
# Load data
X = np.load(os.path.join(TRAIN_TEST_DIR, 'ae_train.npy'))  # (N, T, P)
y = np.load(os.path.join(TRAIN_TEST_DIR, 'genres_train.npy')) # (N,)
print(f"Loaded {X.shape[0]} segments of length {X.shape[1]} with {X.shape[2]} pitch bins.")

### EDA Block 1-12 (Combined Summary)
Here we execute the 12 analysis routines: Pitch Classes, Polyphony, Genre Range, Note Density, Transition Matrices, Centrality, IOI, Melodic Intervals, Correlation, Chord Complexity, and Self-Similarity.

In [ ]:
# Block 1: Genre Distribution
plt.figure(figsize=(8, 3))
genre_names = [GENRES[gi] for gi in y]
sns.countplot(x=genre_names, hue=genre_names, legend=False, palette='bright')
plt.title('EDA 1: Genre Distribution')
plt.show()

In [ ]:
# Block 2: Pitch Class Frequency
pitch_counts = X.sum(axis=(0, 1))
pitch_classes = np.zeros(12)
for i in range(len(pitch_counts)): pitch_classes[(i + 21) % 12] += pitch_counts[i]
plt.figure(figsize=(8, 3))
sns.barplot(x=['C','C#','D','D#','E','F','F#','G','G#','A','A#', 'B'], y=pitch_classes, palette='husl')
plt.title('EDA 2: Pitch Class Distribution')
plt.show()

In [ ]:
# Block 3: Note Polyphony
plt.figure(figsize=(8, 3))
polyphony = X.sum(axis=2).flatten()
sns.histplot(polyphony[polyphony > 0], bins=10, kde=True, color='blue')
plt.title('EDA 3: Polyphony Distribution')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Block 4: Pitch Range Boxplots
ranges = []
for i in range(len(X)):
    active = np.where(X[i].sum(axis=0) > 0)[0]
    ranges.append(active.max() - active.min() if len(active) > 0 else 0)
plt.figure(figsize=(8, 3))
sns.boxplot(x=[GENRES[gi] for gi in y], y=ranges, palette='bright')
plt.title('EDA 4: Pitch Range Spread per Genre')
plt.show()

In [ ]:
# Block 5: Density Violins
densities = X.sum(axis=(1, 2))
plt.figure(figsize=(8, 3))
sns.violinplot(x=[GENRES[gi] for gi in y], y=densities, palette='bright')
plt.title('EDA 5: Note Density Violin Plot')
plt.show()

In [ ]:
# Block 6: Transition Matrix
transitions = np.zeros((12, 12))
for seq in X[:100]:
    active_steps = np.where(seq.sum(axis=1) > 0)[0]
    for i in range(len(active_steps)-1):
        p1, p2 = np.where(seq[active_steps[i]] > 0.5)[0][0]%12, np.where(seq[active_steps[i+1]] > 0.5)[0][0]%12
        transitions[p1, p2] += 1
plt.figure(figsize=(6, 5))
sns.heatmap(transitions, cmap='YlGnBu')
plt.title('EDA 6: Pitch Transitions')
plt.show()

In [ ]:
# Block 7: Pitch Centrality (Stripplot)
avg_pitches = [ (np.where(X[i]>0.5)[1].mean()+21) if np.any(X[i]>0.5) else 60 for i in range(len(X)) ]
plt.figure(figsize=(8, 3))
sns.stripplot(x=[GENRES[gi] for gi in y], y=avg_pitches, palette='bright', alpha=0.5)
plt.title('EDA 7: Average Pitch per Genre')
plt.show()

In [ ]:
# Block 8: IOI Distribution
iois = []
for seq in X[:50]:
    active = np.where(seq.sum(axis=1) > 0)[0]
    if len(active) > 1: iois.extend(np.diff(active))
plt.figure(figsize=(8, 3))
sns.histplot(iois, bins=20, color='red')
plt.title('EDA 8: Inter-Onset Intervals (IOI)')
plt.show()

In [ ]:
# Block 9: Melodic Intervals
intervals = []
for seq in X[:50]:
    active = np.where(seq.sum(axis=1) > 0)[0]
    pitches = [np.where(seq[t] > 0.5)[0][0] for t in active]
    if len(pitches) > 1: intervals.extend(np.diff(pitches))
plt.figure(figsize=(8, 3))
sns.histplot(intervals, bins=30, kde=True, color='purple')
plt.title('EDA 9: Melodic Interval Distribution')
plt.show()

In [ ]:
# Block 10: Correlation Range vs Density
df = pd.DataFrame({'range': ranges, 'density': densities, 'genre': [GENRES[gi] for gi in y]})
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x='range', y='density', hue='genre', palette='bright', alpha=0.6)
plt.title('EDA 10: Range vs Density Correlation')
plt.show()

In [ ]:
# Block 11: Chord Complexity
complexity = []
for seq in X[:50]:
    for t in range(seq.shape[0]):
        active = np.where(seq[t] > 0.5)[0]
        if len(active) > 0: complexity.append(len(set(p % 12 for p in active)))
plt.figure(figsize=(8, 3))
sns.countplot(x=complexity, palette='bright')
plt.title('EDA 11: Chord Complexity Analysis')
plt.show()

In [ ]:
# Block 12: Sequence Self-Similarity
plt.figure(figsize=(4, 4))
sns.heatmap(X[0] @ X[0].T, cmap='viridis', cbar=False)
plt.title('EDA 12: Segment Self-Similarity')
plt.show()

## 2. Training Progress (Tasks 1-4)
Visualizing performance across all 4 tasks.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plots = [('ae_loss_curve.png','Task 1'), ('vae_loss_curves.png','Task 2'), ('transformer_loss.png','Task 3'), ('rlhf_results.png','Task 4')]
for i, (f, t) in enumerate(plots):
    p = os.path.join(PLOTS_DIR, f)
    if os.path.exists(p): axes[i//2, i%2].imshow(plt.imread(p)); axes[i//2, i%2].set_title(t)
    axes[i//2, i%2].axis('off')
plt.tight_layout(); plt.show()

## 3. Final Performance Comparison (Table matching Screenshot 3)
Detailed evaluation against Baselines and Neural Models.

In [ ]:
path = os.path.join(PLOTS_DIR, 'final_comparison_table.png')
if os.path.exists(path): 
    display(Image(filename=path))
else: 
    print("Generate results via src.generation.generate_music to see the table.")

## 4. Latent Space Visualization
Confirming unsupervised learning of genre clusters.

In [ ]:
display(Image(filename=os.path.join(PLOTS_DIR, 'vae_clusters_dbscan.png')))